### eNextSUT generator

Run this notebook to generate or update the eNextSUT reference database from Exiobase Hybrid v3.3.18.

This version always produces three exports for the selected `year`:

* `v1.0`: legacy supply-mix update, matching the original notebook.
* `v2.0`: legacy supply-mix plus legacy electricity trade update, matching the original notebook.
* `v2.1`: new MARIO-native pipeline, exported separately under `v2.1/<year>`.

Set `user` and `year` in the first code cell, then run the notebook from top to bottom.

In [1]:
import mario
import yaml
import pandas as pd
import os

from support.ember_remapping import map_ember_to_classification
import warnings
warnings.filterwarnings("ignore")

user = 'LR'   # change this to your username
year = 2025   # change this to the year you want to update the electricity mixes to

with open('paths.yml', 'r') as file: # open the yml file
    paths = yaml.safe_load(file)

paths = paths[user]

---
## v1.0 and v2.0 - legacy pipeline

This section intentionally keeps the old logic for `v1.0` and `v2.0`: manual EMBER remapping, manual supply-market-share update, manual pooled electricity layer, and `shock_calc` with the legacy trade workbook. Only the deprecated pandas `groupby(axis=1)` syntax is written in the pandas >= 3 equivalent form.

Parse raw Exiobase database

In [ ]:
db = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

Aggregating electricity commodities and activities to match EMBER resolution

In [ ]:
db.aggregate("support/aggregate_ee.xlsx", ignore_nan=True)

In [ ]:
# Parse ember electricity generation data, map to exiobase and get electricity mix for a given year 
ee_mix = map_ember_to_classification(
    path = paths['ember'],
    classification = 'EXIO3',
    year = None,
    mode = 'mix',
)

Updating electricity supply mixes (legacy manual loop)

In [ ]:
z = db.z
s = db.s

for region in db.get_index('Region'):
    print(region, end=' ')
    region_latest_year = ee_mix.loc[(region, slice(None), slice(None))].index.get_level_values(0).max()
    mix_year = year if year <= region_latest_year else region_latest_year
    
    new_mix = ee_mix.loc[(region, mix_year, slice(None)), 'Value'].to_frame().sort_index(axis=0)
    new_mix.index = new_mix.index.get_level_values(2)
    
    old_market_share = s.loc[(region, 'Activity', new_mix.index), (region, 'Commodity', 'Electricity')].sum().sum()
    
    s.loc[(region, 'Activity', new_mix.index), (region, 'Commodity', 'Electricity')] = new_mix['Value'].values * old_market_share
    print('done')

z.update(s)

db.update_scenarios('baseline', z=z)
db.reset_to_coefficients('baseline')

Export the v1.0 database

In [ ]:
db.to_txt(
    path = os.path.join(
        paths['export'],
        "v1.0",
        str(year),
        ),
    )

---
## v2.0 - legacy electricity trades

Define the commodities whose trade mixes are updated and add the related legacy pass-through sectors as empty rows and columns.

In [ ]:
traded_commodities = ['Electricity']

for commodity in traded_commodities:
    new_activities = [f"{commodity} supply"]
    new_commdodities = [f"{commodity} need"]

db.add_sectors(
    new_sectors = new_activities,
    regions = db.get_index("Region"),
    io ="support/add_sectors_activities.xlsx",
    item = "Activity",
    inplace = True,
)

db.add_sectors(
    new_sectors = new_commdodities,
    regions = db.get_index("Region"),
    io ="support/add_sectors_commodities.xlsx",
    item = "Commodity",
    inplace = True,
)

### Demand-side shock

1. Activities supplying the new commodities consume only the domestic original commodity.
2. The consumption of the original commodity is transferred to domestic `need` commodity consumption, both for intermediate use and final demand.

In [ ]:
u_new = db.u.copy()
Y_new = db.Y.copy()

# pandas >= 3 equivalent of the original groupby(level=[0], axis=1).sum()
U = db.U.copy().loc[(slice(None), "Commodity", traded_commodities), :].T.groupby(level=0).sum().T
Y = Y_new.loc[(slice(None), "Commodity", traded_commodities), :].T.groupby(level=0).sum().T
UY = U + Y

z_new = db.z.copy()

trades_df = {}

for commodity in traded_commodities:
    trades_df[commodity] = pd.DataFrame()
    u_new.loc[:, (slice(None), "Activity", f"{commodity} supply")] *= 0
    oth_activities = [i for i in db.get_index("Activity") if i != f"{commodity} supply"]
    
    for region in db.get_index("Region"):
        u_new.loc[(region, "Commodity", commodity), (region, "Activity", f"{commodity} supply")] = 1

        ee_consumption_u = db.u.loc[(slice(None), "Commodity", commodity), (region, "Activity", oth_activities)].sum(0).to_frame().T
        ee_consumption_u.index = pd.MultiIndex.from_arrays([[region], ["Commodity"], [f"{commodity} need"]], names=db.u.index.names)

        ee_consumption_Y = db.Y.loc[(slice(None), "Commodity", commodity), (region, "Consumption category", slice(None))].sum(0).to_frame().T
        ee_consumption_Y.index = pd.MultiIndex.from_arrays([[region], ["Commodity"], [f"{commodity} need"]], names=db.Y.index.names)

        u_new.update(ee_consumption_u)
        Y_new.update(ee_consumption_Y)

        u_new.loc[(slice(None), "Commodity", commodity), (region, "Activity", oth_activities)] *= 0
        Y_new.loc[(slice(None), "Commodity", commodity), (region, "Consumption category", slice(None))] *= 0

        trades_df[commodity] = pd.concat([
            trades_df[commodity], 
            UY.loc[:, region] / UY.loc[:, region].sum()
        ], axis=1
        )

z_new.update(u_new)

Update baseline scenario and reset database to coefficients.

In [ ]:
db.update_scenarios(scenario='baseline', z=z_new, Y=Y_new)
db.reset_to_coefficients('baseline')

### Supply-side shock

Use the legacy shock workbook to apply the electricity trade update.

In [ ]:
# db.get_shock_excel("support/trades.xlsx"))
db.shock_calc(f"support/trades_{year}.xlsx", z=True, scenario='ee_trades', force_rewrite=True)

Export the v2.0 database

In [ ]:
#%% Export the v2.0 database
db.to_txt(
    path = os.path.join(
        paths['export'],
        "v2.0",
        str(year),
        ),
    scenario = 'ee_trades',
    # flows=True,
    # coefficients=True
    )

# Free memory before building v2.1 from a fresh parse.
del db, z, s, u_new, Y_new, z_new, U, Y, UY

---
## v2.1 - new MARIO-native pipeline

Re-parse and re-aggregate the raw table so `v2.1` is independent from the legacy branch. The native electricity update uses EMBER data directly and handles the EXIOBASE Rest-of-World regions through MARIO's packaged region membership.

In [2]:
db3 = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

# Enables MARIO's EXIOBASE Rest-of-World member-country expansion when using EMBER.
db3.meta.source = 'EXIOBASE Hybrid 3.3.18'

db3.aggregate("support/aggregate_ee.xlsx", ignore_nan=True)

INFO Parser: txt reading SUT flows from /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-eNextGen/eNextAll - Documents/Databases/Exiobase Hybrid 3.3.18 with VA/flows in matrix mode (txt/csv).
INFO Parser: Reading flows from txt files.
INFO Parser: Reading files finished.
INFO Parser: Investigating possible identifiable errors.
INFO Parser: parsing database finished.
INFO Parser: state payload ready with 9 canonical blocks.
INFO Parser: txt state ready for SUT.
INFO Metadata: initialized.
WARNING nan values for the aggregation of Activity for following items ignored
['Cultivation of paddy rice', 'Cultivation of wheat', 'Cultivation of cereal grains nec', 'Cultivation of vegetables, fruit, nuts', 'Cultivation of oil seeds', 'Cultivation of sugar cane, sugar beet', 'Cultivation of plant-based fibers', 'Cultivation of crops nec', 'Cattle farming', 'Pigs farming', 'Poultry farming', 'Meat animals nec', 'Animal products nec', 'Raw milk', 'Wool, silk-worm cocoons', 'Manure 

In [3]:
db3.update_supply_mix(
    "electricity",
    scenario = 'baseline',
    year = year,
    ember_path = paths['ember'],
)

INFO Resolver: resolving u for baseline.
INFO Resolver: trying u via formula build_sut_u_from_U_Xa.
INFO Resolver: resolved u via formula build_sut_u_from_U_Xa.
INFO Resolver: resolving s for baseline.
INFO Resolver: trying s via formula build_sut_s_from_S_Xc.
INFO Resolver: resolved s via formula build_sut_s_from_S_Xc.
INFO Resolver: resolving ea for baseline.
INFO Resolver: trying ea via formula build_sut_ea_from_Ea_Xa.
INFO Resolver: resolved ea via formula build_sut_ea_from_Ea_Xa.
INFO Resolver: resolving ec for baseline.
INFO Resolver: trying ec via formula build_sut_ec_from_Ec_Xc.
INFO Resolver: resolved ec via formula build_sut_ec_from_Ec_Xc.
INFO Resolver: resolving va for baseline.
INFO Resolver: trying va via formula build_sut_va_from_Va_Xa.
INFO Resolver: resolved va via formula build_sut_va_from_Va_Xa.
INFO Resolver: resolving vc for baseline.
INFO Resolver: trying vc via formula build_sut_vc_from_Vc_Xc.
INFO Resolver: resolved vc via formula build_sut_vc_from_Vc_Xc.
INFO D

Export the v2.1 supply-mix-only intermediate if you want to inspect it in memory. The formal `v2.1` export is written after the trade mix update below.

Pool the trade of the selected commodities. MARIO adds the `" - supply"` / `" - need"` pass-through layer and stores the observed trade shares in the supply block market shares.

In [4]:
traded_commodities = ['Electricity']

db3.pool_trade(traded_commodities)

INFO Resolver: resolving Z for baseline.
INFO Resolver: trying Z via concat.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=auto, runtime=solve).
INFO Resolver: resolved Z via concat.
INFO Resolver: resolving Y for baseline.
INFO Resolver: trying Y via concat.
INFO Resolver: resolved Y via concat.
INFO Resolver: resolving V for baseline.
INFO Resolver: trying V via concat.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=auto, runtime=solve).
INFO Resolver: resolved V via concat.
INFO Resolver: resolving E for baseline.
INFO Resolver: trying E via concat.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=auto, runtime=solve).
INFO Resolver: resolved E via concat.
INFO pool_trade: pooled trade layer added for ['Electricity'].


The observed trade mixes now live in the `s` market shares of the need columns. This table is also the expected matrix shape for the new trade-workbook format.

In [5]:
db3.reset_to_coefficients('baseline')
s3 = db3.get_block_as_pandas('s', scenario='baseline')

trades_df = {}
for commodity in traded_commodities:
    supply = db3.meta.pooled_trade_map[commodity]['supply']
    need = db3.meta.pooled_trade_map[commodity]['need']
    trades_df[commodity] = pd.DataFrame(
        s3.loc[(slice(None), 'Activity', supply), (slice(None), 'Commodity', need)].values,
        index = db3.get_index('Region'),
        columns = db3.get_index('Region'),
    )

trades_df['Electricity']

INFO Resolver: resolving u for baseline.
INFO Resolver: trying u via extract.
INFO Resolver: trying u via formula build_sut_u_from_U_Xa.
INFO Resolver: resolved u via formula build_sut_u_from_U_Xa.
INFO Resolver: resolving s for baseline.
INFO Resolver: trying s via extract.
INFO Resolver: trying s via formula build_sut_s_from_S_Xc.
INFO Resolver: resolved s via formula build_sut_s_from_S_Xc.
INFO Resolver: resolving ea for baseline.
INFO Resolver: trying ea via extract.
INFO Resolver: trying ea via formula build_sut_ea_from_Ea_Xa.
INFO Resolver: resolved ea via formula build_sut_ea_from_Ea_Xa.
INFO Resolver: resolving ec for baseline.
INFO Resolver: trying ec via extract.
INFO Resolver: trying ec via formula build_sut_ec_from_Ec_Xc.
INFO Resolver: resolved ec via formula build_sut_ec_from_Ec_Xc.
INFO Resolver: resolving va for baseline.
INFO Resolver: trying va via extract.
INFO Resolver: trying va via formula build_sut_va_from_Va_Xa.
INFO Resolver: resolved va via formula build_sut_v

,CY,ZA,KR,BR,CA,ID,GR,WE,SE,FI,...,MX,AU,RU,GB,EE,DE,WL,IN,CN,LU
CY,0.993987,0.000000,0.0,0.000000,0.0,0.0,0.000005,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
ZA,0.000000,0.967858,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
KR,0.000000,0.000000,1.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000,0.0,0.0,0.000218,0.000000
BR,0.000000,0.000000,0.0,0.993474,0.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
CA,0.000000,0.000000,0.0,0.000000,1.0,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
ID,0.000000,0.000000,0.0,0.000000,0.0,1.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
GR,0.005835,0.000000,0.0,0.000000,0.0,0.0,0.864949,0.020614,0.000000,0.000000,...,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
WE,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.037391,0.920583,0.000000,0.000000,...,0.0,0.0,4.027730e-04,0.000000,0.032571,0.000000,0.0,0.0,0.000000,0.000000
SE,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.896092,0.087112,...,0.0,0.0,0.000000e+00,0.000000,0.000000,0.004506,0.0,0.0,0.000000,0.000000
FI,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.048394,0.695109,...,0.0,0.0,1.028395e-04,0.000000,0.007248,0.000000,0.0,0.0,0.000000,0.000000


### Update trade mixes

`support/trades_{year}.xlsx` can be either:

* one origins-by-destinations matrix per commodity, with one sheet named after the commodity; or
* the old MARIO shock workbook with a `z` sheet.

The old workbook is converted on the fly, so the same file used by the legacy `v2.0` branch works for `v2.1` too. `rescale=True` normalizes each destination mix while preserving the destination column totals.

In [6]:
def read_trade_mix(workbook, commodity):
    if commodity in workbook.sheet_names:
        trades = workbook.parse(sheet_name=commodity, index_col=0)
    else:
        legacy = workbook.parse(sheet_name='z')
        legacy = legacy[
            legacy['row sector'].astype(str).eq(f"{commodity} supply")
            & legacy['column sector'].astype(str).eq(f"{commodity} need")
        ]
        if legacy.empty:
            raise ValueError(f"No trade mix found for {commodity!r} in support/trades_{year}.xlsx")
        trades = legacy.pivot_table(
            index='row region',
            columns='column region',
            values='value',
            aggfunc='sum',
        )

    trades.index = trades.index.astype(str)
    trades.columns = trades.columns.astype(str)
    return trades.apply(pd.to_numeric, errors='coerce')

scenario = 'ee_trades'
if scenario not in db3.scenarios:
    db3.clone_scenario('baseline', scenario)

with pd.ExcelFile(f"support/trades_{year}.xlsx") as workbook:
    for commodity in traded_commodities:
        pooled = db3.meta.pooled_trade_map[commodity]
        trades = read_trade_mix(workbook, commodity)

        db3.update_trade_mix(
            {destination: trades[destination].dropna().to_dict() for destination in trades.columns},
            items = pooled['supply'],
            commodities = pooled['need'],
            scenario = scenario,
            rescale = True,
        )

INFO Databases: reset to coefficients.


Export the v2.1 database. The directory is created if it does not exist.

In [7]:
v21_path = os.path.join(paths['export'], "v2.1", str(year))
os.makedirs(v21_path, exist_ok=True)

db3.to_txt(
    path = v21_path,
    scenario = 'ee_trades',
    )

INFO Export: writing txt database for ee_trades in matrix mode.
INFO Resolver: resolving U for ee_trades.
INFO Resolver: trying U via extract.
INFO Resolver: trying U via formula build_sut_U_from_u_Xa.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=auto, runtime=solve).
INFO Resolver: resolved U via formula build_sut_U_from_u_Xa.
INFO Resolver: resolving S for ee_trades.
INFO Resolver: trying S via extract.
INFO Resolver: trying S via formula build_sut_S_from_s_Xc.
INFO Resolver: resolved S via formula build_sut_S_from_s_Xc.
INFO Resolver: resolving Va for ee_trades.
INFO Resolver: trying Va via extract.
INFO Resolver: trying Va via formula build_sut_Va_from_va_Xa.
INFO Resolver: resolved Va via formula build_sut_Va_from_va_Xa.
INFO Resolver: resolving Vc for ee_trades.
INFO Resolver: trying Vc via extract.
INFO Resolver: trying Vc via formula build_sut_Vc_from_vc_Xc.
INFO Resolver: resolved Vc via formula build_sut_Vc_from_vc_Xc.
INFO Resolver: resolvi

---
## Footprint comparison: v2.0 legacy vs v2.1 native

Both versions come from this same run. `v2.0` is read back from its export, so the comparison includes the legacy txt round-trip on that side. The old pooled labels are renamed to the new `" - "` convention before alignment.

In [ ]:
db_old = mario.parse_from_txt(
    path = os.path.join(paths['export'], "v2.0", str(year), "flows"),
    mode = "flows",
    table = 'SUT',
)

In [ ]:
gwp = {
    "Carbon dioxide, fossil (air - Emiss)": 1.0,
    "CH4 (air - Emiss)": 25.0,
    "N2O (air - Emiss)": 298.0,
}

def ghg_footprint(f):
    f = f.loc[list(gwp), :].T
    return sum(f[substance] * factor for substance, factor in gwp.items())

f_new = ghg_footprint(db3.query('f', scenarios='ee_trades'))
f_old = ghg_footprint(db_old.f)

# Align the pooled labels of the legacy pipeline to the new naming convention.
item_renames = {}
for commodity in traded_commodities:
    pooled = db3.meta.pooled_trade_map[commodity]
    item_renames[f"{commodity} supply"] = pooled['supply']
    item_renames[f"{commodity} need"] = pooled['need']
f_old = f_old.rename(index=item_renames, level='Item')

comparison = pd.concat([f_old.rename('v2.0'), f_new.rename('v2.1')], axis=1)
comparison['Delta%'] = 100 * (comparison['v2.1'] / comparison['v2.0'] - 1)

comparison['Delta%'].abs().describe()

In [ ]:
# Largest relative deviations across all items.
comparison.loc[comparison['Delta%'].abs().sort_values(ascending=False).index].head(15)

In [ ]:
# The key check: GHG intensity of the pooled electricity market, per region.
for commodity in traded_commodities:
    need = db3.meta.pooled_trade_map[commodity]['need']
    display(comparison.loc[(slice(None), 'Commodity', need), :].round(4))

In [ ]:
# Quick localization of the largest differences by region.
comparison.assign(abs_delta=comparison['Delta%'].abs()).groupby(level='Region')['abs_delta'].max().sort_values(ascending=False).head(10)